# ULTRA Zero-Shot Inference on MIND splits_updated

Runs ULTRA (`mgalkin/ultra_50g`) zero-shot on all 5 slices of `splits_updated`.
No fine-tuning — pure transfer from pretrained checkpoint.

**Output:** `predictions_test.tsv` and `predictions_valid.tsv` per slice,
same format as RotatE/TransE predictions (drug, expected_disease, rank, top-k diseases).

In [ ]:
# ── Install dependencies ──────────────────────────────────────────────────
import subprocess, sys, os

def run(cmd):
    result = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    if result.returncode != 0:
        print(result.stderr[-2000:])
    else:
        print(result.stdout[-500:] or 'OK')

# PyG — match installed torch version
import torch
tv = torch.__version__.split('+')[0]
cuda = 'cu' + torch.version.cuda.replace('.', '') if torch.cuda.is_available() else 'cpu'
pyg_url = f'https://data.pyg.org/whl/torch-{tv}+{cuda}.html'
print(f'PyTorch {tv}, CUDA tag {cuda}')

run('pip install -q torch_geometric')
run(f'pip install -q torch_scatter torch_sparse torch_cluster -f {pyg_url}')

# Clone ULTRA repo (checkpoint bundled in ULTRA/ckpts/)
if not os.path.exists('/kaggle/working/ULTRA'):
    run('git clone --quiet https://github.com/DeepGraphLearning/ULTRA.git /kaggle/working/ULTRA')
else:
    print('ULTRA already cloned.')

# NOTE: we do NOT patch layers.py.
# ULTRA's rspmm extension is memory-efficient (O(nodes) not O(edges)).
# It compiles JIT via cpp_extension.load() and supports both CPU and GPU.
# Bypassing it causes OOM on graphs with 19M+ edges.
sys.path.insert(0, '/kaggle/working/ULTRA')
print('Done.')

In [ ]:
# ── Imports ───────────────────────────────────────────────────────────────
import os, torch, numpy as np, pandas as pd
from pathlib import Path
from collections import defaultdict
from tqdm import tqdm
from torch_geometric.data import Data

# Detect whether the GPU is actually usable with this PyTorch build.
# Kaggle P100 is sm_60; PyTorch 2.2+ dropped sm_60 support.
def _gpu_usable():
    if not torch.cuda.is_available():
        return False
    try:
        torch.zeros(1, device='cuda')  # lightweight test op
        return True
    except Exception as e:
        print(f'GPU detected but not usable: {e}')
        return False

DEVICE = 'cuda' if _gpu_usable() else 'cpu'
print(f'Device: {DEVICE}')
print(f'PyTorch: {torch.__version__}')
if DEVICE == 'cpu':
    print('NOTE: Running on CPU. Inference will be slower (~3-5 hrs for 5 slices).')

In [ ]:
# ── Config ────────────────────────────────────────────────────────────────
SLICES_TO_RUN = list(range(5))   # run all 5 slices
TOP_K         = 50
BATCH_SIZE    = 64               # reduce if OOM

# Find splits_updated dataset
BASE_INPUT = Path('/kaggle/input')
splits_root = None
for p in BASE_INPUT.rglob('slice_0'):
    if (p / 'ind_test.tsv').exists():
        splits_root = p.parent
        break

assert splits_root is not None, 'Could not find splits_updated dataset'
print(f'Splits found at: {splits_root}')
for sl in range(5):
    files = [f.name for f in (splits_root / f'slice_{sl}').iterdir()]
    print(f'  slice_{sl}: {files}')

OUT_BASE = Path('/kaggle/working/predictions/ULTRA')
OUT_BASE.mkdir(parents=True, exist_ok=True)

In [ ]:
# ── Locate ULTRA checkpoint (bundled in repo) ─────────────────────────────
ckpt_path = '/kaggle/working/ULTRA/ckpts/ultra_50g.pth'
assert os.path.exists(ckpt_path), f'Checkpoint not found: {ckpt_path}'
print(f'Checkpoint: {ckpt_path}')

In [ ]:
# ── Load ULTRA model ──────────────────────────────────────────────────────
from ultra.models import Ultra as UltraModel

# Config matching ULTRA/config/transductive/inference.yaml
rel_model_cfg = dict(
    class_name='RelNBFNet',
    input_dim=64,
    hidden_dims=[64, 64, 64, 64, 64, 64],
    message_func='distmult',
    aggregate_func='sum',
    short_cut=True,
    layer_norm=True,
)
entity_model_cfg = dict(
    class_name='EntityNBFNet',
    input_dim=64,
    hidden_dims=[64, 64, 64, 64, 64, 64],
    message_func='distmult',
    aggregate_func='sum',
    short_cut=True,
    layer_norm=True,
)

# Ultra.__init__ pops 'class' key to select subclass
rel_cfg    = {('class' if k == 'class_name' else k): v for k, v in rel_model_cfg.items()}
entity_cfg = {('class' if k == 'class_name' else k): v for k, v in entity_model_cfg.items()}

model = UltraModel(rel_model_cfg=rel_cfg, entity_model_cfg=entity_cfg)

state = torch.load(ckpt_path, map_location='cpu')
model.load_state_dict(state['model'])
model = model.to(DEVICE)
model.eval()
print('ULTRA model loaded.')
print(f'Parameters: {sum(p.numel() for p in model.parameters()):,}')

In [ ]:
# ── Graph building helpers ────────────────────────────────────────────────

def load_triples(path):
    """Load TSV triples file, return list of (h, r, t) strings."""
    df = pd.read_csv(path, sep='\t', header=None, names=['h','r','t'])
    return list(zip(df['h'], df['r'], df['t']))

def build_graph(triples, entity2id, rel2id):
    """Build PyG Data object from triples with forward + inverse edges."""
    src, dst, edge_type = [], [], []
    num_rels = len(rel2id)

    for h, r, t in triples:
        hid = entity2id[h]
        tid = entity2id[t]
        rid = rel2id[r]
        # Forward edge
        src.append(hid);  dst.append(tid);  edge_type.append(rid)
        # Inverse edge
        src.append(tid);  dst.append(hid);  edge_type.append(rid + num_rels)

    edge_index = torch.tensor([src, dst], dtype=torch.long)
    edge_type_t = torch.tensor(edge_type, dtype=torch.long)

    data = Data(
        edge_index=edge_index,
        edge_type=edge_type_t,
        num_nodes=len(entity2id),
        num_relations=num_rels * 2,
    )
    return data

def build_relation_graph(data):
    """Build relation graph required by ULTRA."""
    from ultra.tasks import build_relation_graph as _brg
    return _brg(data)

print('Helpers defined.')

In [ ]:
# ── Evaluation helper ─────────────────────────────────────────────────────

@torch.no_grad()
def score_queries(model, graph, queries, all_disease_ids, known_pairs, batch_size=BATCH_SIZE):
    """
    For each (drug, relation) query, score all disease candidates using ULTRA.

    ULTRA triple order is [head, tail, relation] — NOT [head, relation, tail].
      batch[:, 0, 2]           → query relation
      h_index, t_index, r_index = batch.unbind(-1)  → positions 0, 1, 2

    We run one Bellman-Ford pass per drug by building batch (1, num_diseases, 3)
    where every row is [drug_id, disease_id, ind_rel_id].
    """
    results = []
    graph = graph.to(DEVICE)

    disease_id_list  = [d_id  for d_id,  _ in all_disease_ids]
    disease_str_list = [d_str for _,  d_str in all_disease_ids]

    for i in tqdm(range(0, len(queries), batch_size), desc='Scoring'):
        batch_q = queries[i:i+batch_size]

        for q in batch_q:
            drug_str = q['drug_str']
            exp_str  = q['exp_str']
            h_id     = q['h_id']
            r_id     = q['r_id']

            # ULTRA expects (bs, num_cands, 3) with order [head, tail, relation]
            batch_tensor = torch.tensor(
                [[[h_id, d_id, r_id] for d_id in disease_id_list]],
                dtype=torch.long, device=DEVICE
            )  # (1, num_diseases, 3)

            scores = model(graph, batch_tensor)  # (1, num_diseases)
            sc = scores[0].cpu().float()         # (num_diseases,)

            # Filtered ranking
            disease_scores = {}
            for d_str, s in zip(disease_str_list, sc.tolist()):
                if (drug_str, d_str) in known_pairs and d_str != exp_str:
                    continue
                disease_scores[d_str] = s

            ranked = sorted(disease_scores, key=disease_scores.get, reverse=True)
            rank   = ranked.index(exp_str) + 1 if exp_str in ranked else len(ranked) + 1

            results.append({
                'drug': drug_str,
                'expected_disease': exp_str,
                'rank': rank,
                'reciprocal_rank': 1.0 / rank,
                **{f'top{k+1}_disease': ranked[k] if k < len(ranked) else ''
                   for k in range(TOP_K)}
            })

    return results

print('Evaluation helper defined.')

In [ ]:
# ── Main loop: run all slices ──────────────────────────────────────────────
import numpy as np

all_results = {}  # (sl, split) -> metrics

for sl in SLICES_TO_RUN:
    print(f'\n{"="*50}\nULTRA zero-shot — slice_{sl}\n{"="*50}')
    slice_dir = splits_root / f'slice_{sl}'

    # Load KGE training graph (full background graph for reasoning)
    train_triples = load_triples(slice_dir / 'kge_train.tsv')

    # Build entity/relation vocabularies from training graph
    entities = sorted(set(h for h,r,t in train_triples) | set(t for h,r,t in train_triples))
    relations = sorted(set(r for h,r,t in train_triples))
    entity2id = {e: i for i, e in enumerate(entities)}
    rel2id    = {r: i for i, r in enumerate(relations)}
    id2entity = {i: e for e, i in entity2id.items()}

    print(f'  Entities: {len(entity2id):,}  Relations: {len(rel2id)}')

    # Indication relation ID
    ind_rel    = 'indication'
    ind_rel_id = rel2id.get(ind_rel, 0)

    # Identify disease nodes (tail entities in indication triples)
    ind_triples = load_triples(slice_dir / 'ind_train.tsv')
    disease_strs = sorted(set(t for h, r, t in ind_triples))
    all_disease_ids = [(entity2id[d], d) for d in disease_strs if d in entity2id]
    print(f'  Disease candidates: {len(all_disease_ids):,}')

    # Build PyG graph
    graph = build_graph(train_triples, entity2id, rel2id)
    graph = build_relation_graph(graph)
    print(f'  Graph edges: {graph.edge_index.shape[1]:,}')

    # Known pairs for filtered evaluation
    known_pairs = set()
    for split in ['train', 'test', 'valid']:
        for h, r, t in load_triples(slice_dir / f'ind_{split}.tsv'):
            known_pairs.add((h, t))

    # Output directory
    out_dir = OUT_BASE / f'slice_{sl}'
    out_dir.mkdir(parents=True, exist_ok=True)

    for split in ['test', 'valid']:
        print(f'\n  -- {split} --')
        ind_df = pd.read_csv(slice_dir / f'ind_{split}.tsv',
                             sep='\t', header=None, names=['h','r','t'])

        # Build queries — only drugs that exist in entity vocab
        queries = []
        skipped = 0
        for _, row in ind_df.iterrows():
            drug, disease = row['h'], row['t']
            if drug not in entity2id or disease not in entity2id:
                skipped += 1
                continue
            queries.append({
                'h_id':     entity2id[drug],
                'r_id':     ind_rel_id,
                't_id':     entity2id[disease],
                'drug_str': drug,
                'exp_str':  disease,
            })
        print(f'  Queries: {len(queries)} (skipped {skipped} missing entities)')

        # Score
        rows = score_queries(model, graph, queries, all_disease_ids, known_pairs)
        df_out = pd.DataFrame(rows)

        # Metrics
        ranks = np.array(df_out['rank'].tolist())
        rrs   = 1.0 / ranks
        N     = len(ranks)
        m = {
            'N':      N,
            'MRR':    round(float(np.mean(rrs)), 4),
            'Hits@1': round(float((ranks<=1).mean()), 4),
            'Hits@3': round(float((ranks<=3).mean()), 4),
            'Hits@5': round(float((ranks<=5).mean()), 4),
            'Hits@10':round(float((ranks<=10).mean()), 4),
        }
        all_results[(sl, split)] = m
        print(f'  {split}: N={N} MRR={m["MRR"]} H@1={m["Hits@1"]} H@3={m["Hits@3"]} H@5={m["Hits@5"]} H@10={m["Hits@10"]}')

        # Save
        out_path = out_dir / f'predictions_{split}.tsv'
        df_out.to_csv(out_path, sep='\t', index=False)
        print(f'  Saved: {out_path}')

print('\nAll slices done.')

In [ ]:
# ── Summary ───────────────────────────────────────────────────────────────
import numpy as np

print('\n' + '='*60)
print('ULTRA Zero-Shot — Summary across all slices')
print('='*60)

for split in ['test', 'valid']:
    vals = {m: [all_results[(sl,split)][m] for sl in range(5) if (sl,split) in all_results]
            for m in ['MRR','Hits@1','Hits@3','Hits@5','Hits@10']}
    if not vals['MRR']: continue
    print(f'\n{split.upper()} SET:')
    print(f'  MRR:     {np.mean(vals["MRR"]):.4f} ± {np.std(vals["MRR"]):.4f}')
    print(f'  Hits@1:  {np.mean(vals["Hits@1"]):.4f} ± {np.std(vals["Hits@1"]):.4f}')
    print(f'  Hits@3:  {np.mean(vals["Hits@3"]):.4f} ± {np.std(vals["Hits@3"]):.4f}')
    print(f'  Hits@5:  {np.mean(vals["Hits@5"]):.4f} ± {np.std(vals["Hits@5"]):.4f}')
    print(f'  Hits@10: {np.mean(vals["Hits@10"]):.4f} ± {np.std(vals["Hits@10"]):.4f}')

print('\nPer-slice results:')
for sl in range(5):
    for split in ['test','valid']:
        k = (sl, split)
        if k in all_results:
            m = all_results[k]
            print(f'  slice_{sl} {split}: MRR={m["MRR"]} H@10={m["Hits@10"]}')

In [ ]:
# ── Zip output for download ───────────────────────────────────────────────
import shutil
shutil.make_archive('/kaggle/working/ultra_predictions', 'zip', '/kaggle/working/predictions')
print('Zipped: /kaggle/working/ultra_predictions.zip')